# Single Layer Case

In [6]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
#from czt import time2freq, freq2time
from scipy.optimize import curve_fit


In [33]:
epsilon_0 = 8.854E-12 # F/m
mu_0 = np.pi*4.0E-7 # H/m
c = 3.0E08 # m/s
lam1 = 450e-9+0.0j
h1 = 10e-6
n0 = 1.0+0.0j
n1 = 1.5+0.0j
e_x = np.array([1.0+0.0j,0.0+0.0j,0.0+0.0j])
e_y = np.array([0.0+0.0j,1.0+0.0j,0.0+0.0j])
e_z = np.array([0.0+0.0j,0.0+0.0j,1.0+0.0j])

norm_vec = np.array([0,0,1])

def TtoS(mat):  # to scattering matrix, diagonal terms are reflection coefficients.
    [[m11,m12],[m21,m22]]=mat
    [[r12,t21],[t12,r21]]=[[-m21/m22,1/m22],[m11-m12*m21/m22,m12/m22]]
    return np.array([[r12,t21],[t12,r21]])

def StoT(co):  # to transfer matrix, 
    [[r12,t21],[t12,r21]]=co
    [[m11,m12],[m21,m22]]=[[(t12- r12*r21/t21),r21/t21],[-r12/t21,1/t21]]
    return np.array([[m11,m12],[m21,m22]])

def interface_s(ni,nj,theta=0):  # scattering matrix, s-polarised, diagonal terms are reflection coefficients.
    phi_ij = np.arcsin(ni/nj*np.sin(theta)) # Snell's law
    phi_ji = np.arcsin(nj/ni*np.sin(theta))
    if np.abs(ni/nj*np.sin(theta))>1:
        print("Total internal reflection going ij")
    if np.abs(nj/ni*np.sin(theta))>1:
        print("Total internal reflection going ji")
    rij = (np.cos(theta)-nj/ni*np.cos(phi_ij))/(np.cos(theta)+nj/ni*np.cos(phi_ij)) # Going ij
    rji = (np.cos(theta)-ni/nj*np.cos(phi_ji))/(np.cos(theta)+ni/nj*np.cos(phi_ji)) # Going ji
    #tij = 2.0*np.sin(phi_ij)*np.cos(theta)/np.sin(theta+phi_ij)
    tij = 1+rij
    tji = 1+rji
    #print(np.abs(rij)**2+nj/ni*np.cos(phi_ij)/np.cos(theta)*np.abs(tij)**2)
    #print(np.abs(rji)**2+ni/nj*np.cos(phi_ji)/np.cos(theta)*np.abs(tji)**2)
    return np.array([[rij,tji],[tij,rji]])

# def interface_s2(ni,nj,theta=0):  # scattering matrix, s-polarised, diagonal terms are reflection coefficients.
#     rij = (np.cos(theta)-np.sqrt(complex((ni/nj)**2-np.sin(theta)**2)))/(np.cos(theta)+np.sqrt(complex((ni/nj)**2-np.sin(theta)**2)))
#     rji = (np.cos(theta)-np.sqrt(complex((nj/ni)**2-np.sin(theta)**2)))/(np.cos(theta)+np.sqrt(complex((nj/ni)**2-np.sin(theta)**2)))
#     tij = 2.0*np.cos(theta)/(np.cos(theta)+np.sqrt(complex((nj/ni)**2-np.sin(theta)**2)))
#     tji = 2.0*np.cos(theta)/(np.cos(theta)+np.sqrt(complex((ni/nj)**2-np.sin(theta)**2)))
#     #print(np.abs(rij)**2+np.sqrt(complex((nj/ni)**2-np.sin(theta)**2))/np.cos(theta)*np.abs(tij)**2)
#     #print(np.abs(rji)**2+np.sqrt(complex((ni/nj)**2-np.sin(theta)**2))/np.cos(theta)*np.abs(tji)**2)
#     return np.array([[rij,tji],[tij,rji]])

def interface_p(ni,nj,theta=0):  # scattering matrix, p-polarised, diagonal terms are reflection coefficients.
    phi_ij = np.arcsin(ni/nj*np.sin(theta))
    phi_ji = np.arcsin(nj/ni*np.sin(theta))
    if np.abs(ni/nj*np.sin(theta))>1:
        print("Total internal reflection going ij")
    if np.abs(nj/ni*np.sin(theta))>1:
        print("Total internal reflection going ji")
    rij = (nj*np.cos(theta)-ni*np.cos(phi_ij))/(nj*np.cos(theta)+ni*np.cos(phi_ij))
    rji = (ni*np.cos(theta)-nj*np.cos(phi_ji))/(ni*np.cos(theta)+nj*np.cos(phi_ji))
    tij = 2.0*ni*np.cos(theta)/(nj*np.cos(theta)+ni*np.cos(phi_ij))
    tji = 2.0*nj*np.cos(theta)/(ni*np.cos(theta)+nj*np.cos(phi_ji))
    #print(np.abs(rij)**2+nj/ni*np.cos(phi_ij)/np.cos(theta)*np.abs(tij)**2) # Going ij
    #print(np.abs(rji)**2+ni/nj*np.cos(phi_ji)/np.cos(theta)*np.abs(tji)**2) # Going ji
    return np.array([[rij,tji],[tij,rji]])

# def interface_p2(ni,nj,theta=0):  # scattering matrix, p-polarised, diagonal terms are reflection coefficients.
#     rij = (nj**2*np.cos(theta)-ni**2*np.sqrt(complex((nj/ni)**2-np.sin(theta)**2)))/(nj**2*np.cos(theta)+ni**2*np.sqrt(complex((nj/ni)**2-np.sin(theta)**2)))
#     rji = (ni**2*np.cos(theta)-nj**2*np.sqrt(complex((ni/nj)**2-np.sin(theta)**2)))/(ni**2*np.cos(theta)+ni**2*np.sqrt(complex((ni/nj)**2-np.sin(theta)**2)))
#     tij = 2.0*ni*nj*np.cos(theta)/(nj**2*np.cos(theta)+ni**2*np.sqrt(complex((nj/ni)**2-np.sin(theta)**2)))
#     tji = 2.0*ni*nj*np.cos(theta)/(ni**2*np.cos(theta)+nj**2*np.sqrt(complex((ni/nj)**2-np.sin(theta)**2)))
#     #print(np.abs(rij)**2+np.sqrt(complex((nj/ni)**2-np.sin(theta)**2))/np.cos(theta)*np.abs(tij)**2)
#     #print(np.abs(rji)**2+np.sqrt(complex((ni/nj)**2-np.sin(theta)**2))/np.cos(theta)*np.abs(tji)**2)
#     return np.array([[rij,tji],[tij,rji]])

def medium(n,lam,h):  # scattering matrix, diagonal terms are reflection coefficients.
    k=2.0*np.pi/lam
    rij=0
    rji=0
    tij=np.exp(1j*n*k*h)
    tji=np.exp(1j*n*k*h)
    return np.array([[rij,tji],[tij,rji]])

def interface_s3(kzi,kzj):  # scattering matrix, s-polarised, diagonal terms are reflection coefficients.
    rij = (kzi - kzj)/(kzi + kzj)
    rji = -rij
    tij = 1 + rij #2*kzi/(kzi + kzj)
    tji = 1 + rji #2*kzj/(kzi + kzj)
    print(1/kzi*(kzi*np.abs(rij)**2+kzj*np.abs(tij)**2))
    print(1/kzj*(kzj*np.abs(rji)**2+kzi*np.abs(tji)**2))
    return np.array([[rij,tji],[tij,rji]])

def interface_s4(ki,ni,nj):  # scattering matrix, s-polarised, diagonal terms are reflection coefficients.
    kzi = ki[2]
    kzj = np.sqrt(kzi**2+np.dot(ki,ki)*((nj/ni)**2-1))
    if np.iscomplex(kzj):
        raise ValueError("Total internal reflection going ij")
    rij = (kzi - kzj)/(kzi + kzj)
    rji = -rij
    tij = 1 + rij #2*kzi/(kzi + kzj)
    tji = 1 + rji #2*kzj/(kzi + kzj)
    #print(1/kzi*(kzi*np.abs(rij)**2+kzj*np.abs(tij)**2))
    #print(1/kzj*(kzj*np.abs(rji)**2+kzi*np.abs(tji)**2))
    return np.array([[rij,tji],[tij,rji]])

def interface_p4(ki,ni,nj):  # scattering matrix, p-polarised, diagonal terms are reflection coefficients.
    kzi = ki[2]
    kzj = np.sqrt(kzi**2+np.dot(ki,ki)*((nj/ni)**2-1))
    if np.iscomplex(kzj):
        raise ValueError("Total internal reflection going ij")
    rij = ((nj/ni)*kzi - kzj)/((nj/ni)*kzi + kzj)
    rji = -rij
    tij = 2*kzi/((nj/ni)*kzi + (ni/nj)*kzj)
    tji = kzj/kzi * tij # 2*kzj/((ni/nj)*kzj + (nj/ni)*kzi)
    # print(1/kzi*(kzi/ni*np.abs(rij)**2+kzj/nj*np.abs(tij)**2)) # add factors of ni nj figure this out
    # print(1/kzj*(kzj*np.abs(rji)**2+kzi*np.abs(tji)**2))
    return np.array([[rij,tji],[tij,rji]])

def medium4(k_vec,n,h):  # scattering matrix, diagonal terms are reflection coefficients.
    # k_vec is a 3D vector
    k = np.sqrt(np.dot(k_vec,k_vec))
    rij=0
    rji=0
    l = h*k/k_vec[2] # kz = k cos (theta)
    tij=np.exp(1j*n*k*l)
    tji=np.exp(1j*n*k*l)
    return np.array([[rij,tji],[tij,rji]])

def interface_s5(ki,kj):  # scattering matrix, s-polarised, diagonal terms are reflection coefficients.
    kzi = ki[2]
    kzj = kj[2]
    if np.iscomplex(kzj):
        raise ValueError("Total internal reflection going ij")
    rij = (kzi - kzj)/(kzi + kzj)
    rji = -rij
    tij = 1 + rij #2*kzi/(kzi + kzj)
    tji = 1 + rji #2*kzj/(kzi + kzj)
    #print(1/kzi*(kzi*np.abs(rij)**2+kzj*np.abs(tij)**2))
    #print(1/kzj*(kzj*np.abs(rji)**2+kzi*np.abs(tji)**2))
    return np.array([[rij,tji],[tij,rji]])

def interface_p5(ki,kj):  # scattering matrix, p-polarised, diagonal terms are reflection coefficients.
    kzi = ki[2]
    kzj = kj[2]
    ki_a = np.dot(ki,ki.conj())
    kj_a = np.dot(kj,kj.conj())
    if np.iscomplex(kzj):
        raise ValueError("Total internal reflection going ij")
    rij = (kzi/ki_a - kzj/kj_a)/(kzi/ki_a + kzj/kj_a)
    rji = -rij
    tij = 2*kzi/((kj_a/ki_a)*kzi + (ki_a/kj_a)*kzj)
    tji = kzj/kzi * tij # 2*kzj/((ni/nj)*kzj + (nj/ni)*kzi)
    # print(1/kzi*(kzi/ni*np.abs(rij)**2+kzj/nj*np.abs(tij)**2)) # add factors of ni nj figure this out
    # print(1/kzj*(kzj*np.abs(rji)**2+kzi*np.abs(tji)**2))
    return np.array([[rij,tji],[tij,rji]])
    
def totalST(layers):
    t=np.identity(2)
    for lay in layers:
        t0=StoT(lay)
        t=t0@t
    s=TtoS(t)
    return s,t

def calc_Es_layer(E_amp_vec,n1,theta=0,lam=lam1, h1=h1):
    # s polarised
    layer_s = [interface_s(n0,n1,theta),medium(n1,lam,h1),interface_s(n1,n0,theta)]
    s0_s,t0_s=totalST(layer_s)
    trans_s = s0_s[1,0]
    ref_s = s0_s[0,0]
    E1_s = E_amp_vec + E_amp_vec*ref_s
    E2_s = trans_s*E_amp_vec
    print(np.abs(ref_s)**2+np.abs(trans_s)**2)
    # p polarised
    layer_p = [interface_p(n0,n1,theta),medium(n1,lam,h1),interface_p(n1,n0,theta)]
    s0_p,t0_p=totalST(layer_p)
    trans_p = s0_p[1,0]
    ref_p = s0_p[0,0]
    E1_p = E_amp_vec + E_amp_vec*ref_p
    E2_p = trans_p*E_amp_vec
    #print(np.abs(ref_p)**2+np.abs(trans_p)**2)
    return E_amp_vec, E1_s, E2_s, E1_p, E2_p

def calc_Es_layer4(E_avi,ki,nj,ni=n0,h=h1):
    E_avsi = E_avi[1] * e_y
    E_avpi = E_avi[0] * np.cross(ki/np.sqrt(np.dot(ki,ki)),e_y)
    #print(E_avsi+E_avpi)
    kj = k_ij(ki,ni,nj)
    # s polarised
    layer_s = [interface_s4(ki,ni,nj),medium4(kj,nj,h),interface_s4(kj,ni,nj)]
    s0_s,t0_s=totalST(layer_s)
    trans_s = s0_s[1,0]
    ref_s = s0_s[0,0]
    E1_s = E_avsi + E_avsi*ref_s
    E2_s = trans_s*E_avsi
    #print(np.abs(ref_s)**2+np.abs(trans_s)**2)
    # p polarised
    layer_p = [interface_p4(ki,ni,nj),medium4(kj,nj,h),interface_p4(kj,ni,nj)]
    s0_p,t0_p=totalST(layer_p)
    trans_p = s0_p[1,0]
    ref_p = s0_p[0,0]
    E1_p = E_avpi + E_avpi*ref_p
    E2_p = trans_p*E_avpi
    #print(np.abs(ref_p)**2+np.abs(trans_p)**2)
    E1 = E1_s + E1_p
    E2 = E2_s + E2_p
    return E_avi, E1, E2

def calc_Es_layer5(E_avi,ki,nj,ni=n0,h=h1):
    E_avsi = E_avi[1] * e_y
    E_avpi = E_avi[0] * np.cross(ki/np.sqrt(np.dot(ki,ki)),e_y)
    #print(E_avsi+E_avpi)
    kj = k_ij(ki,ni,nj)
    # s polarised
    layer_s = [interface_s5(ki,kj),medium4(kj,nj,h),interface_s5(kj,ki)]
    s0_s,t0_s=totalST(layer_s)
    trans_s = s0_s[1,0]
    ref_s = s0_s[0,0]
    E1_s = E_avsi + E_avsi*ref_s
    E2_s = trans_s*E_avsi
    #print(np.abs(ref_s)**2+np.abs(trans_s)**2)
    # p polarised
    layer_p = [interface_p5(ki,kj),medium4(kj,nj,h),interface_p5(kj,ki)]
    s0_p,t0_p=totalST(layer_p)
    trans_p = s0_p[1,0]
    ref_p = s0_p[0,0]
    E1_p = E_avpi + E_avpi*ref_p
    E2_p = trans_p*E_avpi
    #print(np.abs(ref_p)**2+np.abs(trans_p)**2)
    E1 = E1_s + E1_p
    E2 = E2_s + E2_p
    return E_avi, E1, E2

def k_vector(theta, lam):
    k_vec = 2*np.pi/lam*np.array([np.sin(theta),0,np.cos(theta)])
    return k_vec
               
def k_ij(ki,ni,nj):
    kj = np.array([ki[0],ki[1],np.sqrt(ki[2]**2+np.dot(ki,ki)*((nj/ni)**2-1))])
    return kj

def gen_E_amp(E_par,E_perp):
    return np.array([complex(E_par),complex(E_perp),0.0+0.0j])

mapping = {'i': 0, 'sri': 1, 'st': 2, 'pri': 3, 'pt': 4}

def stress_tensor(E_amp_vec, str_input, theta = 0, lam = lam1, n1 = n1, n = n0, mu = mu_0): # Should I add arguments for r_vec and t?
    k_vec = k_vector(theta, lam)
    omega = np.sqrt((c/n)**2*np.dot(k_vec, k_vec))
    E_av = calc_Es_layer(E_amp_vec,n1,theta,lam)[mapping[str_input]]
    H_amp_vec = -1/(mu*omega)*np.cross(k_vec, E_av)
    epsilon = epsilon_0*mu_0/(n**2*mu)
    E_ij = epsilon*np.tensordot(E_av.conj(),E_av,axes=0) - \
    0.5 * epsilon * np.dot(E_av.conj(),E_av) * np.eye(3)
    H_ij = mu*np.tensordot(H_amp_vec.conj(),H_amp_vec,axes=0) - \
    0.5 * mu * np.dot(H_amp_vec.conj(),H_amp_vec) * np.eye(3)
    return E_ij + H_ij

mapping4 = {'i': 0, 'ri': 1, 't': 2}

def stress_tensor4(E_avi,ki,nj,str_input,ni=n0,h=h1): # We always evaluate in vacuum
    omega = np.sqrt(c**2*np.dot(ki, ki))
    E_av = calc_Es_layer4(E_avi,ki,nj,ni,h)[mapping4[str_input]]
    H_av = -1/(mu_0*omega)*np.cross(ki, E_av)
    E_ij = 0.5 * epsilon_0 * (np.tensordot(E_av.conj(),E_av,axes=0) + np.tensordot(E_av,E_av.conj(),axes=0)) - \
    0.5 * epsilon_0 * np.dot(E_av.conj(),E_av) * np.eye(3)
    H_ij = 0.5 * mu_0 * (np.tensordot(H_av.conj(),H_av,axes=0) + np.tensordot(H_av,H_av.conj(),axes=0)) - \
    0.5 * mu_0 * np.dot(H_av.conj(),H_av) * np.eye(3)
    return E_ij + H_ij

def stress_tensor5(E_avi,ki,nj,str_input,ni=n0,h=h1): # We always evaluate in vacuum
    omega = np.sqrt(c**2*np.dot(ki, ki))
    E_av = calc_Es_layer5(E_avi,ki,nj,ni,h)[mapping4[str_input]]
    H_av = -1/(mu_0*omega)*np.cross(ki, E_av)
    E_ij = 0.5 * epsilon_0 * (np.tensordot(E_av.conj(),E_av,axes=0) + np.tensordot(E_av,E_av.conj(),axes=0)) - \
    0.5 * epsilon_0 * np.dot(E_av.conj(),E_av) * np.eye(3)
    H_ij = 0.5 * mu_0 * (np.tensordot(H_av.conj(),H_av,axes=0) + np.tensordot(H_av,H_av.conj(),axes=0)) - \
    0.5 * mu_0 * np.dot(H_av.conj(),H_av) * np.eye(3)
    return E_ij + H_ij

In [74]:
calc_Es_layer4(gen_E_amp(0.5,1),k_vector(0, lam1),n1)

(array([0.5+0.j, 1. +0.j, 0. +0.j]),
 array([-0.5       +0.00000000e+00j,  0.61538462-2.08785109e-14j,
         0.        +0.00000000e+00j]),
 array([-0.32      -1.88184978e-14j,  0.61538462+3.34056174e-14j,
         0.        +0.00000000e+00j]))

In [24]:
interface_p4(k_vector(np.pi/6, lam1),n0,6)

(0.09492512999059824+0j)
(0.545006402396105+0j)


array([[-0.07006337+0.j,  1.93103342+0.j],
       [ 0.27969352+0.j,  0.07006337-0.j]])

In [36]:
stress_tensor5(gen_E_amp(0.5,1),k_vector(np.pi/6, lam1),n1,'t')#@norm_vec

array([[-2.50692502e-12+0.j, -3.92055597e-15+0.j, -4.33328080e-12+0.j],
       [-3.92055597e-15+0.j,  5.10418088e-15+0.j,  2.26353405e-15+0.j],
       [-4.33328080e-12+0.j,  2.26353405e-15+0.j, -7.51056669e-12+0.j]])

The refraction plane is the x-z plane, i.e. an s-polarised wave has the E field pointing along y and a p-polarised wave has the E field pointing along x.

In [ ]:
def show_single_layer(n0,n1,h1):#,width):
    plt.figure()
    plt.xlim(0,11*h1)
    points_per_inch = plt.rcParams['figure.dpi']
    width = (h1 / 2.54) * points_per_inch
    plt.axvline(x=11/2*h1, color='blue', linestyle='-', linewidth = 0.8*width, label = 'Dielectric Medium = ' + str(n1))
    plt.axvline(x=11/2*h1-0.5*h1, color='black', linestyle='-', label = 'Interface 12')
    plt.axvline(x=11/2*h1+0.5*h1, color='black', linestyle='-', label = 'Interface 21')
    plt.plot()
    
def plot_filled_rectangle(rect_width, central_position, rect_height, edgecolor='black', facecolor='lightblue', linewidth=1, label=None):
    left = central_position - rect_width / 2
    bottom = -rect_height/2
    rectangle = patches.Rectangle((left, bottom), rect_width, rect_height, linewidth=linewidth, edgecolor=edgecolor, facecolor=facecolor, label=label)
    plt.gca().add_patch(rectangle)
    
def show_single_layer2(n0,n1,h1,length):
    plt.figure()
    plt.title('Layer system')
    plt.xlabel('Position (m)')
    plt.ylabel('Position (m)')
    plt.xlim(0,length*h1)
    plot_filled_rectangle(h1,length/2*h1,10,label='Dielectric Medium n = ' + str(n1))
    plt.legend()
    plt.show()